In [1]:
!pip install langchain langchain-groq langchain-huggingface langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/

In [3]:
!pip install langchain-text-splitters

In [7]:
import os
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Which files belong to which department
DEPARTMENT_FILES = {
    "engineering": ["engineering_master_doc.md"],
    "finance": ["financial_summary.md", "quarterly_financial_report.md"],
    "hr_dept": ["employee_handbook.md", "hr_data.csv"],
    "marketing": [
        "marketing_report_2024.md",
        "marketing_report_q1_2024.md",
        "marketing_report_q2_2024.md",
        "marketing_report_q3_2024.md",
        "market_report_q4_2024.md"
    ],
    "general": ["employee_handbook.md"]
}

def load_file(filename):
    if filename.endswith(".csv"):
        loader = CSVLoader(filename)
    else:
        loader = TextLoader(filename, encoding="utf-8")
    return loader.load()

# Setup
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create one ChromaDB collection per department
for department, files in DEPARTMENT_FILES.items():
    print(f"Processing {department}...")
    all_docs = []
    for filename in files:
        docs = load_file(filename)
        all_docs.extend(docs)

    chunks = splitter.split_documents(all_docs)

    Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=department,
        persist_directory="./chroma_db"
    )
    print(f"  Done — {len(chunks)} chunks stored")

print("\nAll departments loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing engineering...
  Done — 84 chunks stored
Processing finance...
  Done — 65 chunks stored
Processing hr_dept...
  Done — 141 chunks stored
Processing marketing...
  Done — 109 chunks stored
Processing general...
  Done — 41 chunks stored

All departments loaded successfully!


In [9]:
!pip install -U langchain-chroma


In [11]:
from langchain_chroma import Chroma
# Test: ask a finance question and see if it retrieves the right chunks
query = "What is the total revenue?"

retriever = Chroma(
    collection_name="finance",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
).as_retriever(search_kwargs={"k": 3})

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()


--- Chunk 1 ---
- **Revenue**: $2.6 billion, up 35% YoY, fueled by holiday campaigns and enterprise client acquisitions.
- **Gross Margin**: 64%, reflecting optimized pricing and operational efficiencies.
- **Operating Income**: $650 million, supported by strong revenue and cost discipline.
- **Net Income**: $325 million, up 18% YoY, driven by top-line growth and margin expansion.
- **Marketing Spend**: $650 million, allocated to end-of-year promotions and B2B marketing campaigns.

--- Chunk 2 ---
- **Revenue**: $2.6 billion, up 35% YoY, fueled by holiday campaigns and enterprise client acquisitions.
- **Gross Margin**: 64%, reflecting optimized pricing and operational efficiencies.
- **Operating Income**: $650 million, supported by strong revenue and cost discipline.
- **Net Income**: $325 million, up 18% YoY, driven by top-line growth and margin expansion.
- **Marketing Spend**: $650 million, allocated to end-of-year promotions and B2B marketing campaigns.

--- Chunk 3 ---
- **Revenu

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplateo
from langchain_core.output_parsers import StrOutputParser

GROQ_API_KEY = "Your_API_Key_Here"

# Ask a finance question
question = "What is the total revenue?"
collections = ["finance"]  # Finance Team only sees finance collection

# Get relevant chunks
all_docs = []
for col in collections:
    db = Chroma(collection_name=col, embedding_function=embeddings, persist_directory="./chroma_db")
    retriever = db.as_retriever(search_kwargs={"k": 3})
    all_docs.extend(retriever.invoke(question))

context = "\n\n".join([doc.page_content for doc in all_docs])

# Send to LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

prompt = ChatPromptTemplate.from_template("""
You are an internal assistant for FinSolve Technologies.
Answer the question using only the context provided below.
If the answer is not in the context, say: "I don't have information about that in your accessible documents."

Context:
{context}

Question: {question}

Answer:
""")

chain = prompt | llm | StrOutputParser()
answer = chain.invoke({"context": context, "question": question})
print(answer)

The total revenue is $2.6 billion + $2.1 billion. 

$2.6 billion + $2.1 billion = $4.7 billion.


In [14]:
app_code = '''
import os
import streamlit as st
from langchain_community.document_loaders import TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GROQ_API_KEY = st.secrets["GROQ_API_KEY"]

DEPARTMENT_FILES = {
    "engineering": ["data/engineering/engineering_master_doc.md"],
    "finance": ["data/finance/financial_summary.md", "data/finance/quarterly_financial_report.md"],
    "hr_dept": ["data/hr/employee_handbook.md", "data/hr/hr_data.csv"],
    "marketing": [
        "data/marketing/marketing_report_2024.md",
        "data/marketing/marketing_report_q1_2024.md",
        "data/marketing/marketing_report_q2_2024.md",
        "data/marketing/marketing_report_q3_2024.md",
        "data/marketing/market_report_q4_2024.md"
    ],
    "general": ["data/general/employee_handbook.md"]
}

ROLE_COLLECTIONS = {
    "Finance Team":      ["finance"],
    "HR Team":           ["hr_dept"],
    "Marketing Team":    ["marketing"],
    "Engineering Team":  ["engineering"],
    "C-Level Executive": ["engineering", "finance", "hr_dept", "marketing", "general"],
    "Employee":          ["general"]
}

@st.cache_resource
def load_vectorstores():
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    vectorstores = {}
    for department, files in DEPARTMENT_FILES.items():
        all_docs = []
        for filepath in files:
            if filepath.endswith(".csv"):
                loader = CSVLoader(filepath)
            else:
                loader = TextLoader(filepath, encoding="utf-8")
            all_docs.extend(loader.load())
        chunks = splitter.split_documents(all_docs)
        vectorstores[department] = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=department
        )
    return vectorstores

def get_answer(question, role, vectorstores):
    collections = ROLE_COLLECTIONS[role]
    all_docs = []
    for col in collections:
        retriever = vectorstores[col].as_retriever(search_kwargs={"k": 3})
        all_docs.extend(retriever.invoke(question))
    context = "\\n\\n".join([doc.page_content for doc in all_docs])
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)
    prompt = ChatPromptTemplate.from_template("""
You are an internal assistant for FinSolve Technologies.
Answer the question using only the context provided below.
If the answer is not in the context, say: "I don\'t have information about that in your accessible documents."

Context:
{context}

Question: {question}

Answer:
""")
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question})

# --- Page Config ---
st.set_page_config(page_title="FinSolve Chatbot", page_icon="🏦", layout="wide")

# --- Sidebar ---
st.sidebar.title("🏦 FinSolve Technologies")
st.sidebar.markdown("Internal Knowledge Assistant")
st.sidebar.markdown("---")
st.sidebar.markdown("### How it works")
st.sidebar.markdown("""
- Select your role to login
- Ask questions about your department
- Only your allowed data is searched
- Powered by RAG + LLaMA
""")
st.sidebar.markdown("---")
st.sidebar.markdown("### Tech Stack")
st.sidebar.markdown("""
- 🦙 LLaMA 3.1 (Groq)
- 🔗 Langchain
- 🗄️ ChromaDB
- 🤗 HuggingFace Embeddings
- 🎨 Streamlit
""")
st.sidebar.markdown("---")
st.sidebar.markdown("Built by **Mohammad Murtaza**")
st.sidebar.markdown("[GitHub Profile](https://github.com/Murtaza-data)")

# --- Session State ---
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "role" not in st.session_state:
    st.session_state.role = None
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- Login Screen ---
if not st.session_state.logged_in:
    st.title("🏦 FinSolve Technologies")
    st.subheader("Internal Knowledge Assistant")
    st.markdown("### Built by Mohammad Murtaza | RAG + RBAC + LLaMA + Langchain")
    st.markdown("---")

    col1, col2, col3 = st.columns(3)
    with col1:
        st.info("🔐 Role-based access control")
    with col2:
        st.info("🗄️ Department-specific data")
    with col3:
        st.info("🤖 AI-powered answers")

    st.markdown("---")
    st.markdown("### Select Your Role to Login")
    role = st.selectbox("", list(ROLE_COLLECTIONS.keys()))
    if st.button("🔐 Login", type="primary", use_container_width=True):
        st.session_state.logged_in = True
        st.session_state.role = role
        st.session_state.messages = []
        st.rerun()

# --- Chat Screen ---
else:
    col1, col2 = st.columns([4, 1])
    with col1:
        st.title("🏦 FinSolve Chatbot")
        st.markdown(f"Logged in as: **{st.session_state.role}**")
    with col2:
        st.markdown("###")
        if st.button("🚪 Logout", use_container_width=True):
            st.session_state.logged_in = False
            st.session_state.role = None
            st.session_state.messages = []
            st.rerun()

    st.markdown("---")

    vectorstores = load_vectorstores()

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.write(message["content"])

    if prompt := st.chat_input("Ask a question about your department..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.write(prompt)
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                answer = get_answer(prompt, st.session_state.role, vectorstores)
            st.write(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})

    st.markdown("---")
    st.markdown(
        "Built by **Mohammad Murtaza** | "
        "[GitHub](https://github.com/Murtaza-data) | "
        "Powered by RAG + RBAC + LLaMA + Langchain"
    )
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!
